# VQA B2 — DPO (Direct Preference Optimization) với PaliGemma

**DPO là gì?**  
Thay vì dùng PPO (phức tạp, cần reward model), DPO học trực tiếp từ cặp `(chosen, rejected)`.  
Model được huấn luyện để tăng xác suất sinh `chosen` và giảm xác suất sinh `rejected`.

**Công thức DPO loss:**
```
L_DPO = -log σ( β · (log π_θ(chosen)/π_ref(chosen) - log π_θ(rejected)/π_ref(rejected)) )
```
- `β` nhỏ → model tự do học preference hơn  
- `β` lớn → model bám sát SFT gốc hơn

## CELL 1 — Cài đặt thư viện

In [ ]:
# Pin version để tránh breaking changes giữa các bản TRL
# - TRL >= 0.11: DPOConfig + processing_class (multimodal vision-language)
# - peft >= 0.11: hỗ trợ PaliGemma LoRA target modules
!pip install -q "transformers>=4.45,<4.50" "trl>=0.11,<0.13" "peft>=0.11" \
                "datasets>=2.20" "accelerate>=0.34" "bitsandbytes>=0.43" \
                "bert-score" "rouge-score" "nltk"
print("[✓] Cài đặt thư viện xong.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00


## CELL 2 — Import

In [ ]:
import os, json, torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from dataclasses import dataclass
from google.colab import drive

from datasets import Dataset
from transformers import (
    PaliGemmaForConditionalGeneration,
    PaliGemmaProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

# TRL multimodal DPO API:
#  - DPOConfig: kế thừa TrainingArguments + bao gồm beta, max_length, max_prompt_length
#  - DPOTrainer: dùng `processing_class=processor` thay vì `tokenizer=...` (TRL ≥ 0.11)
# Pin version trong CELL 1 để tránh breaking changes giữa TRL 0.7/0.8/0.11+
from trl import DPOConfig, DPOTrainer

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_scorer_lib
from bert_score import score as bert_score_fn

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("[✓] Import hoàn tất.")

## CELL 3 — Kiểm tra GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị: {device}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"    GPU  : {torch.cuda.get_device_name(0)}")
    print(f"    VRAM : {vram_gb:.1f} GB")
    if vram_gb < 14:
        print("[!] CẢNH BÁO: VRAM < 14GB. Nên dùng Colab A100.")

drive.mount("/content/drive")

MAIN_ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
MAIN_EXTRACT  = "/content/data"
DATASET_ROOT  = os.path.join(MAIN_EXTRACT, "dataset_7_5_26")

if not os.path.isdir(DATASET_ROOT):
    os.makedirs(MAIN_EXTRACT, exist_ok=True)
    print("[*] Giải nén dataset...")
    os.system(f'unzip -q "{MAIN_ZIP_PATH}" -d "{MAIN_EXTRACT}"')
    print("[✓] Xong.")
else:
    print("[✓] Dataset đã tồn tại.")

# ── Preference JSONL ─────────────────────────────────────────────────────────
# DATASET 300: đã clean qua nhiều vòng review (Sonnet + ChatGPT 5.5),
# 90 classes × 3-5 mẫu/class, 43% câu hỏi visual-grounded
#
# LUÔN overwrite từ Drive để tránh dùng bản cũ trong cùng Colab session.
# Verify số dòng = 300 trước khi chạy tiếp.
PREF_JSONL_LOCAL  = "/content/preference_300.jsonl"
PREF_JSONL_REMOTE = "/content/drive/MyDrive/RL_data/preference_300.jsonl"

assert os.path.exists(PREF_JSONL_REMOTE), (
    f"Không tìm thấy {PREF_JSONL_REMOTE} trên Drive.\n"
    f"→ Upload preference_300.jsonl vào /MyDrive/RL_data/ trước."
)

# Force overwrite mỗi lần chạy cell
os.system(f'cp -f "{PREF_JSONL_REMOTE}" "{PREF_JSONL_LOCAL}"')

# Verify số dòng
with open(PREF_JSONL_LOCAL, "r", encoding="utf-8") as f:
    n_lines = sum(1 for line in f if line.strip())
assert n_lines == 300, f"Pref file phải có 300 dòng, thực tế: {n_lines}"

print(f"[✓] Preference data: {PREF_JSONL_LOCAL} ({n_lines} dòng)")

In [ ]:
drive.mount("/content/drive")

MAIN_ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
MAIN_EXTRACT  = "/content/data"
DATASET_ROOT  = os.path.join(MAIN_EXTRACT, "dataset_7_5_26")

if not os.path.isdir(DATASET_ROOT):
    os.makedirs(MAIN_EXTRACT, exist_ok=True)
    print("[*] Giải nén dataset...")
    os.system(f'unzip -q "{MAIN_ZIP_PATH}" -d "{MAIN_EXTRACT}"')
    print("[✓] Xong.")
else:
    print("[✓] Dataset đã tồn tại.")

# Copy file preference JSONL về local để đọc nhanh hơn
# DATASET 300: đã clean qua nhiều vòng review (Sonnet + ChatGPT 5.5),
# 90 classes × 3-5 mẫu/class, 43% câu hỏi visual-grounded
PREF_JSONL_LOCAL  = "/content/preference_300.jsonl"
PREF_JSONL_REMOTE = "/content/drive/MyDrive/RL_data/preference_300.jsonl"
if not os.path.exists(PREF_JSONL_LOCAL):
    os.system(f'cp "{PREF_JSONL_REMOTE}" "{PREF_JSONL_LOCAL}"')
assert os.path.exists(PREF_JSONL_LOCAL), f"Không tìm thấy {PREF_JSONL_REMOTE} trên Drive"
print(f"[✓] Preference data: {PREF_JSONL_LOCAL}")

## CELL 5 — Cấu hình toàn cục

In [ ]:
MODEL_NAME       = "google/paligemma-3b-pt-224"
HF_TOKEN         = ""  # điền nếu model yêu cầu xác thực
SFT_ADAPTER_PATH = "/content/drive/MyDrive/ENDTERM_DEEPLEARNING_OUTPUTS/B2_SFT/checkpoint_final"

# Đường dẫn dữ liệu
PREF_JSONL_PATH = PREF_JSONL_LOCAL
TRAIN_IMAGE_DIR = os.path.join(DATASET_ROOT, "train")
TEST_CSV_PATH   = os.path.join(DATASET_ROOT, "test.csv")
TEST_IMAGE_DIR  = os.path.join(DATASET_ROOT, "test")

OUTPUT_DIR = "/content/drive/MyDrive/ENDTERM_DEEPLEARNING_OUTPUTS/B2_DPO"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Tham số DPO ───────────────────────────────────────────────────────────────
# Beta = 0.1 (default DPO) — dataset 300 mẫu đủ lớn, không cần beta nhỏ như khi 100 mẫu
# Beta nhỏ hơn (0.05) → model tự do học preference hơn
# Beta lớn hơn (0.2-0.3) → model bám sát SFT gốc hơn
DPO_BETA          = 0.1

# Learning rate nhỏ để tránh quên kiến thức từ SFT
DPO_LEARNING_RATE = 5e-6

# A100 80GB → batch lớn được; A100 40GB → giữ batch=2
DPO_BATCH_SIZE    = 2      # per device
DPO_GRAD_ACCUM    = 4      # effective batch = 2×4 = 8

# Dataset 300 mẫu × 3 epochs ≈ 113 steps (với batch effective 8)
# Đủ để model học preference mà không overfit
DPO_EPOCHS        = 3
DPO_MAX_STEPS     = -1     # -1 = chạy đủ epochs; >0 = dừng sớm

DPO_MAX_LENGTH    = 256    # tokens tối đa cho toàn bộ prompt + response
DPO_PROMPT_LENGTH = 128    # tokens tối đa cho phần prompt

# Inference
MAX_NEW_TOKENS  = 32
EVAL_BATCH_SIZE = 4

print("[✓] Cấu hình sẵn sàng.")
print(f"    Beta     : {DPO_BETA}")
print(f"    Epochs   : {DPO_EPOCHS}")
print(f"    LR       : {DPO_LEARNING_RATE}")
print(f"    Batch    : {DPO_BATCH_SIZE} × {DPO_GRAD_ACCUM} = {DPO_BATCH_SIZE * DPO_GRAD_ACCUM} (effective)")
print(f"    Output   : {OUTPUT_DIR}")

## CELL 6 — Load Processor & Model

In [ ]:
# 4-bit quantization: giảm VRAM từ ~24GB xuống ~8GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# ── Verify SFT adapter EXISTS — không fallback im lặng ───────────────────────
# Lý do: nếu chạy DPO trên base model (không có SFT), kết quả "SFT vs DPO"
# thực ra là "base vs DPO", và 528M params sẽ được train (vision encoder)
# thay vì chỉ ~5M LoRA. Đây là lỗi nghiêm trọng cho experiment.
assert SFT_ADAPTER_PATH and os.path.isdir(SFT_ADAPTER_PATH), (
    f"SFT adapter không tồn tại: {SFT_ADAPTER_PATH}\n"
    f"→ Chạy notebook B2_SFT trước để tạo adapter, hoặc:\n"
    f"   • Sửa SFT_ADAPTER_PATH trong CELL 5 cho đúng\n"
    f"   • Hoặc tạo LoRA adapter mới rõ ràng (xem cell tiếp theo nếu cần)"
)

processor = PaliGemmaProcessor.from_pretrained(
    MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None
)

# ── Active Model (sẽ được DPO cập nhật) ──────────────────────────────────────
print("[*] Load active model...")
active_base = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
)
# tắt use_cache để tránh conflict với gradient checkpointing
active_base.config.use_cache = False

active_model = PeftModel.from_pretrained(active_base, SFT_ADAPTER_PATH, is_trainable=True)
print("[✓] Active model: base + SFT LoRA adapter (trainable).")

# bật gradient checkpointing TRƯỚC khi tạo trainer
active_model.gradient_checkpointing_enable()

# ── Reference Model (đóng băng hoàn toàn) ────────────────────────────────────
print("[*] Load reference model...")
ref_base = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
)
ref_base.config.use_cache = False

reference_model = PeftModel.from_pretrained(ref_base, SFT_ADAPTER_PATH, is_trainable=False)

reference_model.eval()
for param in reference_model.parameters():
    param.requires_grad = False

print("[✓] Cả hai model sẵn sàng (active=trainable, reference=frozen).")

In [ ]:
# 4-bit quantization: giảm VRAM từ ~24GB xuống ~8GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = PaliGemmaProcessor.from_pretrained(
    MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None
)

# ── Active Model (sẽ được DPO cập nhật) ──────────────────────────────────────
print("[*] Load active model...")
active_base = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
)
# FIX: tắt use_cache để tránh conflict với gradient checkpointing
active_base.config.use_cache = False

if SFT_ADAPTER_PATH and os.path.isdir(SFT_ADAPTER_PATH):
    active_model = PeftModel.from_pretrained(active_base, SFT_ADAPTER_PATH, is_trainable=True)
    print("[✓] Active model: base + SFT LoRA adapter.")
else:
    print("[!] Không tìm thấy SFT adapter — dùng model gốc.")
    active_model = active_base

# FIX: bật gradient checkpointing TRƯỚC khi tạo trainer
active_model.gradient_checkpointing_enable()

# ── Reference Model (đóng băng hoàn toàn) ────────────────────────────────────
print("[*] Load reference model...")
ref_base = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
)
# FIX: tắt use_cache cho reference model
ref_base.config.use_cache = False

if SFT_ADAPTER_PATH and os.path.isdir(SFT_ADAPTER_PATH):
    reference_model = PeftModel.from_pretrained(ref_base, SFT_ADAPTER_PATH, is_trainable=False)
else:
    reference_model = ref_base

reference_model.eval()
for param in reference_model.parameters():
    param.requires_grad = False

print("[✓] Cả hai model sẵn sàng.")

## CELL 7 — DEBUG: Kiểm tra Trainable Parameters

In [ ]:
# FIX: Verify chỉ LoRA layers được train, vision encoder phải FROZEN
# Nếu vision encoder đang train → VRAM tăng mạnh, DPO unstable

trainable_params = []
frozen_params    = []

for name, param in active_model.named_parameters():
    if param.requires_grad:
        trainable_params.append((name, param.numel()))
    else:
        frozen_params.append((name, param.numel()))

total_trainable = sum(n for _, n in trainable_params)
total_all       = total_trainable + sum(n for _, n in frozen_params)
trainable_pct   = total_trainable / total_all * 100

print(f"Tổng tham số    : {total_all:,}")
print(f"Đang train      : {total_trainable:,}  ({trainable_pct:.2f}%)")
print(f"Đóng băng       : {total_all - total_trainable:,}")
print(f"\n--- Các layer đang train (chỉ nên là LoRA) ---")
for name, num_params in trainable_params[:20]:  # in 20 cái đầu
    print(f"  {name}: {num_params:,}")

# Kiểm tra vision encoder có bị train không
vision_trainable = [name for name, _ in trainable_params if "vision" in name.lower()]
if vision_trainable:
    print(f"\n[!] CẢNH BÁO: Vision encoder đang train! Cần freeze thủ công.")
    print(f"    Các layer: {vision_trainable[:5]}")

    # FIX: Freeze vision encoder thủ công
    for name, param in active_model.named_parameters():
        if "vision" in name.lower():
            param.requires_grad = False
    print("[✓] Đã freeze vision encoder.")
else:
    print("\n[✓] Vision encoder đã frozen. Chỉ LoRA layers đang train.")

def resolve_image_path(raw_image_path: str, image_dir: str) -> str:
    """
    Chuyển đường dẫn trong JSONL sang đường dẫn thực tế trên đĩa.

    Dataset 300 lưu path dạng:  "animals/<class>/<filename>.jpg"
    Trên Drive có thể là một trong các structure:
      A) DATASET_ROOT/train/<class>/<filename>.jpg  (split train/test)
      B) DATASET_ROOT/animals/<class>/<filename>.jpg
      C) /content/data/animals/<class>/<filename>.jpg

    Hàm này thử nhiều candidate đường dẫn để tương thích cả 3 structure.
    """
    raw_path   = raw_image_path.replace("\\", "/")
    label_name = raw_path.split("/")[-2]
    file_name  = raw_path.split("/")[-1]

    # Thử nhiều structure khả dĩ
    candidates = [
        os.path.join(image_dir, label_name, file_name),                 # train/<class>/file
        os.path.join(image_dir, file_name),                              # train/file
        os.path.join(DATASET_ROOT, "animals", label_name, file_name),    # ROOT/animals/<class>/file
        os.path.join(MAIN_EXTRACT, "animals", label_name, file_name),    # /content/data/animals/<class>/file
        os.path.join(MAIN_EXTRACT, raw_path),                            # /content/data/animals/<class>/file (raw)
        raw_image_path,                                                   # nguyên bản từ JSONL
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Không tìm thấy ảnh: {raw_image_path}")


def load_preference_data(jsonl_path: str, image_dir: str, expected_count: int = 300) -> Dataset:
    """
    Đọc JSONL và trả về HuggingFace Dataset.
    Mỗi sample: {image_path, prompt, chosen, rejected}

    STRICT MODE: fail nếu thiếu BẤT KỲ ảnh nào hoặc len != expected_count.
    Lý do: với DPO 300 samples, mất 5-10 mẫu đã làm lệch preference signal đáng kể.
    """
    records       = []
    missing_paths = []

    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line_no, line in enumerate(file, start=1):
            item = json.loads(line.strip())
            try:
                full_image_path = resolve_image_path(item["image"], image_dir)
            except FileNotFoundError:
                missing_paths.append((line_no, item["image"]))
                continue

            # Làm sạch prompt: bỏ prefix "vqa ", thêm "<image> "
            question         = item["prompt"].replace("vqa ", "").strip()
            formatted_prompt = f"<image> {question}"

            records.append({
                "image_path": full_image_path,
                "prompt"    : formatted_prompt,
                "chosen"    : item["chosen"],
                "rejected"  : item["rejected"],
            })

    # ── STRICT validation ────────────────────────────────────────────────────
    if missing_paths:
        msg = [f"[FAIL] Thiếu {len(missing_paths)} ảnh:"]
        for line_no, img in missing_paths[:10]:
            msg.append(f"  L{line_no}: {img}")
        if len(missing_paths) > 10:
            msg.append(f"  ... và {len(missing_paths) - 10} ảnh khác")
        msg.append(f"\n→ Kiểm tra TRAIN_IMAGE_DIR='{image_dir}' và structure dataset trên Drive.")
        raise FileNotFoundError("\n".join(msg))

    assert len(records) == expected_count, (
        f"Expect {expected_count} samples, got {len(records)}"
    )

    print(f"[✓] Load đủ {len(records)} mẫu preference (không thiếu ảnh).")
    return Dataset.from_list(records)


preference_dataset = load_preference_data(PREF_JSONL_PATH, TRAIN_IMAGE_DIR, expected_count=300)

# In ví dụ để kiểm tra
sample = preference_dataset[0]
print(f"\nVí dụ sample:")
print(f"  image_path : {sample['image_path']}")
print(f"  prompt     : {sample['prompt']}")
print(f"  chosen     : {sample['chosen']}")
print(f"  rejected   : {sample['rejected']}")

In [ ]:
def resolve_image_path(raw_image_path: str, image_dir: str) -> str:
    """
    Chuyển đường dẫn trong JSONL sang đường dẫn thực tế trên đĩa.

    Dataset 300 lưu path dạng:  "animals/<class>/<filename>.jpg"
    Trên Drive có thể là một trong các structure:
      A) DATASET_ROOT/train/<class>/<filename>.jpg  (split train/test)
      B) DATASET_ROOT/animals/<class>/<filename>.jpg
      C) /content/data/animals/<class>/<filename>.jpg

    Hàm này thử nhiều candidate đường dẫn để tương thích cả 3 structure.
    """
    raw_path   = raw_image_path.replace("\\", "/")
    label_name = raw_path.split("/")[-2]
    file_name  = raw_path.split("/")[-1]

    # Thử nhiều structure khả dĩ
    candidates = [
        os.path.join(image_dir, label_name, file_name),                  # train/<class>/file
        os.path.join(image_dir, file_name),                              # train/file
        os.path.join(DATASET_ROOT, "animals", label_name, file_name),    # ROOT/animals/<class>/file
        os.path.join(MAIN_EXTRACT, "animals", label_name, file_name),    # /content/data/animals/<class>/file
        os.path.join(MAIN_EXTRACT, raw_path),                            # /content/data/raw_path
        os.path.join(
            "/content/drive/MyDrive/dataanimals",
            label_name,
            file_name
        ),                                                               # Drive/dataanimals/<class>/file
        raw_image_path,                                                  # nguyên bản từ JSONL
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Không tìm thấy ảnh: {raw_image_path}")


def load_preference_data(jsonl_path: str, image_dir: str) -> Dataset:
    """
    Đọc JSONL và trả về HuggingFace Dataset.
    Mỗi sample: {image_path, prompt, chosen, rejected}
    """
    records       = []
    missing_count = 0

    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line in file:
            item = json.loads(line.strip())
            try:
                full_image_path = resolve_image_path(item["image"], image_dir)
            except FileNotFoundError:
                missing_count += 1
                continue

            # Làm sạch prompt: bỏ prefix "vqa ", thêm "<image> "
            question         = item["prompt"].replace("vqa ", "").strip()
            formatted_prompt = f"<image> {question}"

            records.append({
                "image_path": full_image_path,
                "prompt"    : formatted_prompt,
                "chosen"    : item["chosen"],
                "rejected"  : item["rejected"],
            })

    print(f"[✓] Load được {len(records)} mẫu preference.")
    if missing_count > 0:
        print(f"[!] Bỏ qua {missing_count} mẫu do không tìm thấy ảnh.")
        print(f"    → Kiểm tra TRAIN_IMAGE_DIR='{image_dir}' và structure dataset trên Drive.")
    return Dataset.from_list(records)


preference_dataset = load_preference_data(PREF_JSONL_PATH, TRAIN_IMAGE_DIR)
assert len(preference_dataset) > 0, "Dataset rỗng — kiểm tra path ảnh trong CELL 4 & CELL 5."

# In ví dụ để kiểm tra
sample = preference_dataset[0]
print(f"\nVí dụ sample:")
print(f"  image_path : {sample['image_path']}")
print(f"  prompt     : {sample['prompt']}")
print(f"  chosen     : {sample['chosen']}")
print(f"  rejected   : {sample['rejected']}")

## CELL 9 — Custom Multimodal Collator

In [ ]:
@dataclass
class DPOMultimodalCollator:
    """
    Collator tuỳ chỉnh cho DPO với PaliGemma.

    DPOTrainer mặc định chỉ xử lý text. Vì có ảnh, collator này:
    1. Mở ảnh từ đường dẫn
    2. Tokenize {prompt, chosen, rejected} + ảnh qua processor
    3. Mask phần prompt trong labels bằng -100
       (DPO chỉ tính loss trên phần RESPONSE, không phải prompt)
    4. Trả về dict đúng format DPOTrainer
    """
    processor      : PaliGemmaProcessor
    max_length     : int = 256
    prompt_length  : int = 128

    def _tokenize(self, texts: list[str], images: list, max_len: int) -> dict:
        """Helper: tokenize text + ảnh với truncation rõ ràng."""
        return self.processor(
            text           = texts,
            images         = images,
            return_tensors = "pt",
            padding        = "max_length",
            # FIX: truncation phải explicit và nhất quán cho cả chosen/rejected
            truncation     = True,
            max_length     = max_len,
        )

    def _mask_prompt_in_labels(self, labels: torch.Tensor, prompt_lengths: torch.Tensor) -> torch.Tensor:
        """
        FIX: Mask phần prompt trong labels bằng -100.
        Đây là bước quan trọng — nếu không mask, model sẽ tính loss
        cả trên câu hỏi, làm nhiễu DPO loss.
        """
        masked_labels = labels.clone()
        for i, prompt_len in enumerate(prompt_lengths):
            # Mask toàn bộ token từ đầu đến hết prompt
            masked_labels[i, :prompt_len] = -100
        return masked_labels

    def __call__(self, batch: list[dict]) -> dict:
        prompts   = [s["prompt"]     for s in batch]
        chosens   = [s["chosen"]     for s in batch]
        rejecteds = [s["rejected"]   for s in batch]
        images    = [Image.open(s["image_path"]).convert("RGB") for s in batch]

        # ── Tokenize prompt riêng để đo độ dài ─────────────────────────────────
        prompt_encoding  = self._tokenize(prompts, images, self.prompt_length)
        # Đếm số token thực (không tính padding)
        prompt_lengths   = prompt_encoding["attention_mask"].sum(dim=1)

        # ── Tokenize chosen (prompt + chosen answer) ────────────────────────────
        chosen_texts     = [f"{p} {c}" for p, c in zip(prompts, chosens)]
        chosen_encoding  = self._tokenize(chosen_texts, images, self.max_length)

        # ── Tokenize rejected (prompt + rejected answer) ────────────────────────
        rejected_texts    = [f"{p} {r}" for p, r in zip(prompts, rejecteds)]
        rejected_encoding = self._tokenize(rejected_texts, images, self.max_length)

        # ── Tạo labels với prompt được mask ─────────────────────────────────────
        chosen_labels   = self._mask_prompt_in_labels(
            chosen_encoding["input_ids"],  prompt_lengths
        )
        rejected_labels = self._mask_prompt_in_labels(
            rejected_encoding["input_ids"], prompt_lengths
        )

        return {
            # Prompt
            "prompt_input_ids"        : prompt_encoding["input_ids"],
            "prompt_attention_mask"   : prompt_encoding["attention_mask"],
            # Chosen
            "chosen_input_ids"        : chosen_encoding["input_ids"],
            "chosen_attention_mask"   : chosen_encoding["attention_mask"],
            "chosen_labels"           : chosen_labels,
            # Rejected
            "rejected_input_ids"      : rejected_encoding["input_ids"],
            "rejected_attention_mask" : rejected_encoding["attention_mask"],
            "rejected_labels"         : rejected_labels,
            # FIX: pixel_values phải được truyền vào để DPO thực sự dùng ảnh
            "pixel_values"            : chosen_encoding["pixel_values"],
        }


data_collator = DPOMultimodalCollator(
    processor     = processor,
    max_length    = DPO_MAX_LENGTH,
    prompt_length = DPO_PROMPT_LENGTH,
)
print("[✓] Collator sẵn sàng.")

## CELL 10 — DEBUG: Kiểm tra Collator Output

In [ ]:
# FIX: Bắt buộc verify collator trước khi train
# Nếu bỏ qua bước này, DPO có thể chạy text-only mà không báo lỗi

test_batch = data_collator([preference_dataset[0], preference_dataset[1]])

print("=" * 55)
print("  DEBUG: Kiểm tra output của collator")
print("=" * 55)
print(f"Keys trong batch: {list(test_batch.keys())}")
print()

# 1. Verify pixel_values tồn tại và đúng shape
assert "pixel_values" in test_batch, "[FAIL] Thiếu pixel_values!"
pixel_shape = test_batch["pixel_values"].shape
print(f"[✓] pixel_values shape: {pixel_shape}")
print(f"    Kỳ vọng: (batch=2, channels=3, height=224, width=224)")
assert len(pixel_shape) == 4 and pixel_shape[1] == 3, "[FAIL] pixel_values sai shape!"
print()

# 2. Verify label masking: prompt tokens phải là -100
chosen_labels = test_batch["chosen_labels"][0]
num_masked    = (chosen_labels == -100).sum().item()
num_active    = (chosen_labels != -100).sum().item()
print(f"[✓] chosen_labels: {num_masked} tokens bị mask (-100) | {num_active} tokens tính loss")
assert num_masked > 0, "[FAIL] Không có token nào bị mask! Prompt chưa được mask."
assert num_active > 0, "[FAIL] Không có token nào để tính loss! Answer có thể bị truncate."
print()

# 3. In 40 token đầu của labels để verify trực tiếp
print(f"chosen_labels (40 token đầu):")
print(f"  {chosen_labels[:40].tolist()}")
print(f"  (Các số -100 là phần prompt bị mask)")
print()

# 4. Verify không có NaN trong pixel_values
has_nan = torch.isnan(test_batch["pixel_values"]).any()
assert not has_nan, "[FAIL] pixel_values chứa NaN!"
print(f"[✓] pixel_values không chứa NaN.")
print()
print("[✓] TẤT CẢ CHECKS PASSED — Sẵn sàng train DPO.")

## CELL 11 — DEBUG: Kiểm tra Model thực sự dùng Ảnh

In [ ]:
# FIX: Verify image branch không bị ignore
# Test: cùng câu hỏi, đổi ảnh → prediction phải khác nhau
# Nếu prediction giống nhau → model đang text-only

print("[*] Kiểm tra xem model có thực sự xử lý ảnh không...")

active_model.eval()
sample_question = "Đây là con vật gì?"
prompt          = f"<image> {sample_question}"

# Ảnh 1: sample thực từ dataset
image1 = Image.open(preference_dataset[0]["image_path"]).convert("RGB")
# Ảnh 2: ảnh trắng giả (nếu model ignore ảnh → output giống image1)
image2 = Image.new("RGB", (224, 224), color=(255, 255, 255))

def quick_inference(image, question):
    inputs = processor(
        text=f"<image> {question}", images=image,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        output = active_model.generate(**inputs, max_new_tokens=15, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    return processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

answer1 = quick_inference(image1, sample_question)
answer2 = quick_inference(image2, sample_question)

print(f"  Ảnh thực   → '{answer1}'")
print(f"  Ảnh trắng  → '{answer2}'")

if answer1 != answer2:
    print("[✓] Model đang dùng ảnh — output khác nhau khi đổi ảnh.")
else:
    print("[!] CẢNH BÁO: Output giống nhau — có thể image branch đang bị ignore.")
    print("    Kiểm tra lại collator và pixel_values.")

active_model.train()

## CELL 12 — Khởi tạo DPO Trainer

In [ ]:
# =================================================================
# DPO Trainer — TRL multimodal API (>= 0.11)
# =================================================================
# Khác biệt so với TRL cũ:
#  - Dùng DPOConfig (subclass TrainingArguments) thay vì TrainingArguments + beta riêng
#  - DPOTrainer nhận `processing_class=processor` (multimodal) thay vì `tokenizer=...`
#  - beta, max_length, max_prompt_length truyền qua DPOConfig
#
# Vì PaliGemma là vision-language, processor phải đi qua processing_class
# để DPOTrainer biết cách xử lý ảnh + text trong reference forward pass.

dpo_config = DPOConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = DPO_BATCH_SIZE,
    gradient_accumulation_steps = DPO_GRAD_ACCUM,
    learning_rate               = DPO_LEARNING_RATE,
    num_train_epochs            = DPO_EPOCHS,
    max_steps                   = DPO_MAX_STEPS,
    logging_steps               = 5,
    save_steps                  = 50,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "cosine",
    bf16                        = True,
    remove_unused_columns       = False,
    gradient_checkpointing      = True,
    report_to                   = "none",
    # ── DPO-specific ──────────────────────────────────────────────
    beta                        = DPO_BETA,
    max_length                  = DPO_MAX_LENGTH,
    max_prompt_length           = DPO_PROMPT_LENGTH,
)

dpo_trainer = DPOTrainer(
    model            = active_model,
    ref_model        = reference_model,
    args             = dpo_config,
    train_dataset    = preference_dataset,
    processing_class = processor,         # NEW API: multimodal processor (text + image)
    data_collator    = data_collator,     # custom collator handle pixel_values
)
print("[✓] DPO Trainer sẵn sàng (TRL multimodal API).")
print(f"    Tổng steps dự kiến: ~{len(preference_dataset) * DPO_EPOCHS // (DPO_BATCH_SIZE * DPO_GRAD_ACCUM)}")

## CELL 13 — DEBUG: Verify batch từ Trainer DataLoader

In [ ]:
# FIX: Verify batch từ trainer's dataloader (khác với collator test trên)
# DPOTrainer có thể transform lại data → cần check lần nữa

print("[*] Lấy 1 batch từ trainer dataloader...")
train_dataloader = dpo_trainer.get_train_dataloader()
first_batch      = next(iter(train_dataloader))

print(f"Keys trong batch của trainer: {list(first_batch.keys())}")

# Verify pixel_values vẫn còn sau khi qua trainer
if "pixel_values" in first_batch:
    print(f"[✓] pixel_values tồn tại: shape={first_batch['pixel_values'].shape}")
else:
    print("[!] CẢNH BÁO: pixel_values bị mất trong trainer dataloader!")
    print("    DPO sẽ chạy text-only. Kiểm tra remove_unused_columns=False.")

# Verify labels masking vẫn đúng
if "chosen_labels" in first_batch:
    labels    = first_batch["chosen_labels"][0]
    n_masked  = (labels == -100).sum().item()
    n_active  = (labels != -100).sum().item()
    print(f"[✓] Labels: {n_masked} masked | {n_active} active")

print("[✓] Batch check xong. Sẵn sàng train.")

## CELL 14 — Huấn luyện DPO

In [ ]:
print("=" * 60)
print("  BẮT ĐẦU HUẤN LUYỆN DPO")
print(f"  Số mẫu     : {len(preference_dataset)}  (dataset 300 đã clean)")
print(f"  Epochs     : {DPO_EPOCHS}  (đủ để học preference, chưa overfit)")
print(f"  Beta       : {DPO_BETA}")
print(f"  LR         : {DPO_LEARNING_RATE}")
print(f"  Batch eff. : {DPO_BATCH_SIZE * DPO_GRAD_ACCUM}")
print("=" * 60)
print()
print("DPO Trainer mỗi step sẽ:")
print("  1. Tính log P_active(chosen | prompt, image)")
print("  2. Tính log P_active(rejected | prompt, image)")
print("  3. Tính log P_reference(chosen | prompt, image)  [frozen]")
print("  4. Tính log P_reference(rejected | prompt, image) [frozen]")
print("  5. DPO loss = -log σ(β · (ratio_chosen - ratio_rejected))")
print("  6. Backprop chỉ qua LoRA weights")
print()

train_result = dpo_trainer.train()

print(f"\n[✓] Training xong!")
print(f"    Train loss cuối : {train_result.training_loss:.4f}")
print(f"    Tổng steps      : {train_result.global_step}")

## CELL 15 — Lưu Model

In [ ]:
final_model_path = os.path.join(OUTPUT_DIR, "PaliGemma_DPO_Final")
active_model.save_pretrained(final_model_path)
processor.save_pretrained(final_model_path)
print(f"[✓] Model DPO lưu tại: {final_model_path}")

with open(os.path.join(OUTPUT_DIR, "dpo_training_info.json"), "w") as f:
    json.dump({
        "train_loss"   : train_result.training_loss,
        "global_step"  : train_result.global_step,
        "dpo_beta"     : DPO_BETA,
        "learning_rate": DPO_LEARNING_RATE,
        "epochs"       : DPO_EPOCHS,
        "num_samples"  : len(preference_dataset),
    }, f, indent=2)
print("[✓] Training info đã lưu.")

## CELL 16 — Chuẩn bị Inference

In [ ]:
class VQATestDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path: str, image_dir: str):
        self.dataframe = pd.read_csv(csv_path)
        self.image_dir = image_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index: int) -> dict:
        row       = self.dataframe.iloc[index]
        file_name = os.path.basename(str(row["image_path"]))
        label     = str(row["label"])

        candidates = [
            os.path.join(self.image_dir, label, file_name),
            os.path.join(self.image_dir, file_name),
        ]
        image_path = next((p for p in candidates if os.path.exists(p)), candidates[0])

        return {
            "image_path" : image_path,
            "question"   : str(row["question"]),
            "answer"     : str(row["answer"]),
            "label"      : label,
            "q_type"     : str(row["q_type"]),
        }


def collate_batch(batch: list[dict]) -> dict:
    return {key: [s[key] for s in batch] for key in batch[0]}


def run_inference(model_to_eval, images: list, questions: list[str]) -> list[str]:
    prompts = [f"<image> {q}" for q in questions]
    inputs  = processor(
        text=prompts, images=images,
        return_tensors="pt", padding=True
    ).to(device)
    with torch.no_grad():
        output_ids = model_to_eval.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False
        )
    input_length  = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, input_length:]
    return [
        t.strip() for t in
        processor.batch_decode(generated_ids, skip_special_tokens=True)
    ]


test_dataset = VQATestDataset(TEST_CSV_PATH, TEST_IMAGE_DIR)
test_loader  = torch.utils.data.DataLoader(
    test_dataset, batch_size=EVAL_BATCH_SIZE,
    shuffle=False, collate_fn=collate_batch,
)
print(f"[✓] Test dataset: {len(test_dataset)} mẫu | {len(test_loader)} batches")

## CELL 17 — Chạy Inference: SFT vs DPO

In [ ]:
print("[*] Chạy inference SFT vs DPO...")

active_model.eval()
reference_model.eval()
all_results = []

for batch in tqdm(test_loader, desc="Đánh giá"):
    images          = [Image.open(p).convert("RGB") for p in batch["image_path"]]
    dpo_predictions = run_inference(active_model,    images, batch["question"])
    sft_predictions = run_inference(reference_model, images, batch["question"])

    for i in range(len(batch["answer"])):
        all_results.append({
            "question"      : batch["question"][i],
            "ground_truth"  : batch["answer"][i],
            "sft_prediction": sft_predictions[i],
            "dpo_prediction": dpo_predictions[i],
            "label"         : batch["label"][i],
            "q_type"        : batch["q_type"][i],
        })

df_results = pd.DataFrame(all_results)
print(f"[✓] Inference xong: {len(df_results)} mẫu.")

## CELL 18 — Tính Metrics

In [ ]:
def normalize(text: str) -> str:
    return str(text).lower().strip()

def exact_match(df: pd.DataFrame, pred_col: str) -> float:
    return sum(
        normalize(r["ground_truth"]) == normalize(r[pred_col])
        for _, r in df.iterrows()
    ) / len(df)

def semantic_accuracy(df: pd.DataFrame, pred_col: str, threshold: float = 0.85) -> float:
    """
    FIX: Thêm semantic accuracy thay vì chỉ dùng exact match.
    Dùng BERTScore F1 > threshold làm 'đúng'.
    Giải quyết vấn đề paraphrase: 'đây là con dê' vs 'con vật là con dê'
    đều tính là đúng nếu BERTScore > threshold.
    """
    preds = [normalize(r[pred_col])       for _, r in df.iterrows()]
    refs  = [normalize(r["ground_truth"]) for _, r in df.iterrows()]
    _, _, f1 = bert_score_fn(
        preds, refs, lang="vi",
        model_type="bert-base-multilingual-cased",
        device=str(device), verbose=False,
    )
    return (f1 >= threshold).float().mean().item()

def bleu_scores(df: pd.DataFrame, pred_col: str) -> tuple[float, float]:
    smooth    = SmoothingFunction().method1
    b1, b4    = [], []
    for _, r in df.iterrows():
        ref = [normalize(r["ground_truth"]).split()]
        hyp =  normalize(r[pred_col]).split()
        b1.append(sentence_bleu(ref, hyp, weights=(1,0,0,0), smoothing_function=smooth))
        b4.append(sentence_bleu(ref, hyp, weights=(.25,.25,.25,.25), smoothing_function=smooth))
    return np.mean(b1), np.mean(b4)

def rouge_l(df: pd.DataFrame, pred_col: str) -> float:
    scorer = rouge_scorer_lib.RougeScorer(["rougeL"], use_stemmer=False)
    return np.mean([
        scorer.score(normalize(r["ground_truth"]), normalize(r[pred_col]))["rougeL"].fmeasure
        for _, r in df.iterrows()
    ])

def bertscore_f1(df: pd.DataFrame, pred_col: str) -> float:
    preds = [normalize(r[pred_col])       for _, r in df.iterrows()]
    refs  = [normalize(r["ground_truth"]) for _, r in df.iterrows()]
    _, _, f1 = bert_score_fn(
        preds, refs, lang="vi",
        model_type="bert-base-multilingual-cased",
        device=str(device), verbose=False,
    )
    return f1.mean().item()

# ── Tính metrics cho cả SFT và DPO ───────────────────────────────────────────
print("[*] Tính metrics...")
metrics = {}
for model_name, pred_col in [("SFT", "sft_prediction"), ("DPO", "dpo_prediction")]:
    b1, b4 = bleu_scores(df_results, pred_col)
    metrics[model_name] = {
        "Exact Match Accuracy" : round(exact_match(df_results, pred_col),          4),
        "Semantic Accuracy"    : round(semantic_accuracy(df_results, pred_col),     4),  # FIX: thêm metric này
        "BLEU-1"               : round(b1,                                          4),
        "BLEU-4"               : round(b4,                                          4),
        "ROUGE-L"              : round(rouge_l(df_results, pred_col),               4),
        "BERTScore F1"         : round(bertscore_f1(df_results, pred_col),          4),
    }

# ── In bảng so sánh ──────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  KẾT QUẢ SO SÁNH: SFT vs DPO")
print("=" * 65)
print(pd.DataFrame(metrics).T.to_string())

print("\n--- Mức độ cải thiện (DPO - SFT) ---")
for metric in metrics["SFT"]:
    delta = metrics["DPO"][metric] - metrics["SFT"][metric]
    sign  = "↑" if delta >= 0 else "↓"
    print(f"  {metric:<25}: {sign} {abs(delta):.4f}")

## CELL 19 — Phân tích theo Q_Type & Lưu kết quả

In [ ]:
print("=" * 65)
print("  ACCURACY THEO LOẠI CÂU HỎI")
print("=" * 65)
for q_type, group in df_results.groupby("q_type"):
    sft_acc = exact_match(group, "sft_prediction")
    dpo_acc = exact_match(group, "dpo_prediction")
    delta   = dpo_acc - sft_acc
    sign    = "↑" if delta >= 0 else "↓"
    print(f"  {q_type:<20}: SFT={sft_acc*100:.1f}%  DPO={dpo_acc*100:.1f}%  {sign}{abs(delta)*100:.1f}%  ({len(group)} mẫu)")

# Lưu kết quả
df_results.to_csv(os.path.join(OUTPUT_DIR, "B2_DPO_evaluation.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "B2_DPO_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f"\n[✓] Kết quả lưu tại: {OUTPUT_DIR}")

# Xem ví dụ định tính
print("\n--- 5 ví dụ so sánh ---\n")
for _, row in df_results.sample(5, random_state=42).iterrows():
    print(f"  Câu hỏi     : {row['question']}")
    print(f"  Đúng        : {row['ground_truth']}")
    print(f"  SFT dự đoán : {row['sft_prediction']}")
    print(f"  DPO dự đoán : {row['dpo_prediction']}")
    print("-" * 55)